# Pre-Consultation Agent - Kaggle GPU Deployment

This notebook deploys your pre-consultation agent on Kaggle's **FREE Tesla P100 GPU** (better than Colab's T4!).

## Setup Checklist:
1. ✅ Turn on GPU: **Settings → Accelerator → GPU P100**
2. ✅ Enable Internet: **Settings → Internet → ON**
3. ✅ Add Secrets (optional but recommended):
   - Go to **Add-ons → Secrets**
   - Add `GEMINI_API_KEY` and `HF_TOKEN`

**GPU Quota**: 30 hours/week (more generous than Colab!)

Let's get started! 🚀

## Step 1: Verify GPU and Environment

In [ ]:
import torch
import os

print("🔍 Checking environment...\n")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f}GB")
else:
    print("❌ No GPU detected!")
    print("   Go to Settings → Accelerator → GPU P100")

# Check Python version
import sys
print(f"\n🐍 Python: {sys.version.split()[0]}")

# Check working directory
print(f"📁 Working dir: {os.getcwd()}")

## Step 2: Clone Repository from GitHub

In [ ]:
import os
from pathlib import Path

# Remove if already exists
if Path('/kaggle/working/pre-consultation-agent').exists():
    !rm -rf /kaggle/working/pre-consultation-agent

# Clone from GitHub
print("📥 Cloning repository from GitHub...\n")
print("⚠️  Make sure Internet is enabled: Settings → Internet → ON\n")

# Attempt to clone
clone_result = !git clone https://github.com/buwituze/pre-consultation-agent.git /kaggle/working/pre-consultation-agent 2>&1

# Check if clone was successful
repo_path = Path('/kaggle/working/pre-consultation-agent')
backend_path = repo_path / 'backend'

if backend_path.exists():
    # Navigate to backend
    os.chdir(str(backend_path))
    print(f"\n✅ Repository cloned successfully!")
    print(f"📁 Current directory: {os.getcwd()}")
else:
    print("\n❌ Failed to clone repository!")
    print("\n🔍 Troubleshooting:")
    print("   1. Enable Internet: Go to Settings → Internet → ON")
    print("   2. Check if repository exists: https://github.com/buwituze/pre-consultation-agent")
    print("   3. Rerun this cell after enabling Internet")
    print("\n📋 Clone output:")
    for line in clone_result:
        print(f"   {line}")
    raise Exception("Repository clone failed - Internet might be disabled")

In [ ]:
# Pull latest changes from GitHub (if repository already exists)
import os
from pathlib import Path

backend_path = Path('/kaggle/working/pre-consultation-agent/backend')

if backend_path.exists():
    os.chdir('/kaggle/working/pre-consultation-agent')
    print("🔄 Pulling latest changes from GitHub...\n")
    !git pull origin main
    
    print("\n✅ Latest code pulled!")
    print("\n📌 Current commit:")
    !git log -1 --oneline
    print("\n📅 Commit details:")
    !git log -1 --pretty=format:"Commit: %h%nAuthor: %an%nDate: %ar%nMessage: %s"
    
    # Navigate back to backend
    os.chdir(str(backend_path))
    print(f"\n📁 Working directory: {os.getcwd()}")
else:
    print("⚠️ Repository not found. Run the clone cell above first.")

## Step 3: Install Dependencies

This installs all required packages including transformers, torch, librosa, fastapi, etc.

In [ ]:
# Install requirements
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

print("\n✅ Dependencies installed!")

# Verify key packages
import transformers
import librosa
import fastapi
print(f"\n📚 Key versions:")
print(f"   transformers: {transformers.__version__}")
print(f"   librosa: {librosa.__version__}")
print(f"   fastapi: {fastapi.__version__}")

## Step 4: Configure Environment Variables

**⚠️ IMPORTANT**: Add your API keys below OR use Kaggle Secrets (recommended).

### Using Kaggle Secrets (Recommended):
1. Click **Add-ons → Secrets** in the right sidebar
2. Add these secrets:
   - `GEMINI_API_KEY`: Your Google Gemini API key
   - `HF_TOKEN`: Your Hugging Face token
3. Enable them for this notebook

### Manual Setup:
If you don't use Secrets, replace the values below.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Try to get from Kaggle Secrets first
try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("GEMINI_API_KEY")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ Using Kaggle Secrets")
except:
    print("⚠️ Kaggle Secrets not found - using manual setup")
    # Manual setup (replace these!)
    gemini_key = 'YOUR_GEMINI_API_KEY_HERE'
    hf_token = 'YOUR_HUGGINGFACE_TOKEN_HERE'

# Set environment variables
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['HF_TOKEN'] = hf_token
os.environ['DEVICE'] = 'cuda'  # Use GPU!
os.environ['MAX_TURNS'] = '6'

# Verify
print("\n✅ Environment configured:")
print(f"   GEMINI_API_KEY: {'✓ SET' if gemini_key and 'YOUR_' not in gemini_key else '❌ NOT SET'}")
print(f"   HF_TOKEN: {'✓ SET' if hf_token and 'YOUR_' not in hf_token else '❌ NOT SET'}")
print(f"   DEVICE: {os.environ['DEVICE']}")

if 'YOUR_' in gemini_key or 'YOUR_' in hf_token:
    print("\n⚠️ WARNING: Please update API keys before continuing!")

## Step 5: Load Whisper Models

This downloads and loads both Whisper models (~6GB total). **First time takes 3-5 minutes**.

The models will be cached for future runs.

In [ ]:
import sys
import time
sys.path.insert(0, '/kaggle/working/pre-consultation-agent/backend')

from models import model_a

print("🔄 Loading Whisper models on GPU...")
print("   This takes 3-5 minutes on first run (downloading ~6GB)\n")

start = time.time()
model_a.load_models()
status = model_a.get_models_status()
elapsed = time.time() - start

print(f"\n✅ Models loaded in {elapsed:.1f}s")
print(f"   Status: {status}")

# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"\n📊 GPU Memory:")
    print(f"   Allocated: {allocated:.2f}GB")
    print(f"   Reserved: {reserved:.2f}GB")

## Step 6: Test Transcription with Audio File

**How to upload your audio file:**
1. Click **"+ Add Data"** button (right sidebar)
2. Click **"Upload"** → **"New Dataset"**
3. Upload your WAV file (e.g., `Benitha_testfile.wav`)
4. Name your dataset (e.g., "test-audio")
5. The file will be available at: `/kaggle/input/test-audio/Benitha_testfile.wav`

**Expected Performance**: ~5 seconds for 24-second audio on P100 GPU

In [ ]:
import time
import os
import glob

print("📁 Looking for WAV files in uploaded datasets...\n")

# Automatically find WAV files in /kaggle/input/
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if wav_files:
    print(f"✅ Found {len(wav_files)} WAV file(s):")
    for i, filepath in enumerate(wav_files, 1):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   {i}. {os.path.basename(filepath)} ({size_mb:.2f}MB)")
        print(f"      Path: {filepath}")
    
    # Use the first WAV file found
    audio_file_path = wav_files[0]
    print(f"\n🎤 Using: {os.path.basename(audio_file_path)}\n")
    
    # Read audio file
    with open(audio_file_path, 'rb') as f:
        audio_bytes = f.read()
    
    print(f"   Size: {len(audio_bytes)/(1024*1024):.2f}MB")
    
    # Run transcription
    start = time.time()
    result = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
    elapsed = time.time() - start
    
    print(f"\n✅ Transcription completed in {elapsed:.1f}s")
    print(f"   Language: {result['dominant_language']}")
    print(f"   Confidence: {result['mean_confidence']:.2%}")
    print(f"\n📝 Transcription:")
    print(f"   {result['full_text']}")
    
    # GPU memory
    if torch.cuda.is_available():
        print(f"\n📊 GPU Memory Used: {torch.cuda.memory_allocated(0)/1e9:.2f}GB")
else:
    print("❌ No WAV files found in uploaded datasets!")
    print("\n📤 To upload your audio file:")
    print("   1. Click '+ Add Data' button (right sidebar)")
    print("   2. Click 'Upload' → 'New Dataset'")
    print("   3. Upload your WAV file (e.g., Benitha_testfile.wav)")
    print("   4. Name your dataset (e.g., 'test-audio')")
    print("   5. Re-run this cell")
    print("\n💡 Your file will be at: /kaggle/input/<dataset-name>/Benitha_testfile.wav")

## Step 7: Quick Test with Sample Audio (Alternative)

If you don't have a WAV file handy, you can create a simple test file.

In [ ]:
# Generate a simple test WAV file (1 second of silence)
import numpy as np
import soundfile as sf

print("🔧 Creating test audio file...\n")

# Create 1 second of silence at 16kHz
sample_rate = 16000
duration = 1  # seconds
audio = np.zeros(sample_rate * duration, dtype=np.float32)

# Save as WAV
test_file = '/kaggle/working/test_audio.wav'
sf.write(test_file, audio, sample_rate)

# Read it back
with open(test_file, 'rb') as f:
    audio_bytes = f.read()

print(f"📁 Test file created: {len(audio_bytes)} bytes\n")

# Test transcription (will return empty/minimal text for silence)
import time
start = time.time()
result = model_a.transcribe(audio_bytes, language_hint='english')
elapsed = time.time() - start

print(f"✅ Transcription completed in {elapsed:.1f}s")
print(f"   Result: {result}")
print("\nℹ️ Empty/minimal text is expected for silence.")
print("   Upload a real audio file in Step 6 for actual transcription.")

## Step 8: Start FastAPI Server (Local Testing)

This starts the server locally in Kaggle. You won't be able to access it from outside, but you can test the API programmatically.

**Note**: Kaggle doesn't easily expose public URLs like Colab+ngrok. This is mainly for testing that the server runs.

In [ ]:
print("🚀 Starting FastAPI server in background...\n")

# Start server in background
import subprocess
import time

# Kill any existing uvicorn processes
!pkill -f uvicorn

# Start server
process = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/kaggle/working/pre-consultation-agent/backend',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(5)

print("✅ Server starting...")
print("   URL: http://localhost:8000")
print("   Docs: http://localhost:8000/docs")
print("\nℹ️ Server is running in background.")
print("   You can test it programmatically in the next step.")

## Step 9: Test API Programmatically

Since we can't access the server from outside Kaggle, we'll test it programmatically using requests.

In [ ]:
import requests
import time

API_URL = "http://localhost:8000"

print("🧪 Testing API locally...\n")

# Test 1: Health check
print("Step 1: Health check...")
try:
    response = requests.get(f"{API_URL}/")
    print(f"   Status: {response.status_code}")
    print(f"   Response: {response.json()}\n")
except Exception as e:
    print(f"   ❌ Error: {e}\n")

# Test 2: Create session
print("Step 2: Creating session...")
try:
    response = requests.post(f"{API_URL}/kiosk/start")
    data = response.json()
    session_id = data['id']
    print(f"   Status: {response.status_code}")
    print(f"   Session ID: {session_id}")
    print(f"   Greeting: {data['greeting']}\n")
    
    # Test 3: Upload audio (if you have a file)
    test_audio = '/kaggle/working/test_audio.wav'
    if os.path.exists(test_audio):
        print("Step 3: Uploading audio...")
        with open(test_audio, 'rb') as f:
            files = {'file': ('test.wav', f, 'audio/wav')}
            response = requests.post(
                f"{API_URL}/kiosk/{session_id}/audio",
                files=files,
                timeout=60
            )
        print(f"   Status: {response.status_code}")
        print(f"   Response: {response.json()}\n")
        
except Exception as e:
    print(f"   ❌ Error: {e}\n")

print("✅ API testing complete!")

## Summary

**What we've done:**
1. ✅ Verified GPU access (Tesla P100)
2. ✅ Cloned your repository from GitHub
3. ✅ Installed all dependencies
4. ✅ Loaded Whisper models on GPU (~6GB)
5. ✅ Tested transcription (should be ~5s for 24s audio)
6. ✅ Started FastAPI server locally

**Performance:**
- **GPU**: Tesla P100 (16GB) - Better than Colab's T4!
- **Processing Speed**: ~5 seconds for 24-second audio
- **Weekly Quota**: 30 hours/week

**Limitations:**
- Kaggle doesn't easily expose public URLs like ngrok
- Best for testing transcription functionality
- For public API access, use Colab + ngrok or deploy to cloud

**Next Steps:**
- For production, consider Hugging Face Spaces (free) or AWS EC2
- See your `DEPLOYMENT.md` file for full deployment options

Enjoy your FREE GPU transcription! 🎉